In [3]:
import importlib
import sys
import os
import torch
import numpy as np
from tqdm.notebook import tqdm
import torch

sys.path.insert(0, '..')
sys.path.insert(1, '../../..')
sys.path.insert(0, "../../src")  # src package


## Generate Train, Val, Test

In [4]:
from perturbation_logic.activity_pertubator import (
    split_prefix_suffix_readable,
    redo_last_activity_of_prefix)
from event_log_loader_service.event_log_loader import (get_train_test_val_datasets,
                                                       extract_feature_info)
np.random.seed(17)

csv_path="../../../data/BPI Challenge 2020.csv"

properties = {
                        'case_name' : 'case:concept:name',
                        'concept_name' : 'concept:name',
                        'timestamp_name' : 'time:timestamp',
                        'time_since_case_start_column' : 'case_elapsed_time',
                        'time_since_last_event_column' : 'event_elapsed_time',
                        'day_in_week_column' : 'day_in_week',
                        'seconds_in_day_column' : 'seconds_in_day',
                        'min_suffix_size' : 5,
                        'train_validation_size' : 0.15,
                        'test_validation_size' : 0.2,
                        'window_size' : 'auto',
                        'categorical_columns' : ['concept:name', 'org:resource', 'org:role', 'case:BudgetNumber','case:DeclarationNumber'],
                        'continuous_columns' : ['case_elapsed_time', 'event_elapsed_time', 'day_in_week', 'seconds_in_day', ],
                        'continuous_positive_columns' : []
}


train_df, val_df, test_df,  = get_train_test_val_datasets(csv_path, properties)


print(len(train_df))

# data_train = split_prefix_suffix_readable(
#     train_df,
#     case_column=properties["case_name"],
#     activity_column=properties["concept_name"],
#     min_suffix_size=2,
# )

data_val = split_prefix_suffix_readable(
    val_df,
    case_column=properties["case_name"],
    activity_column=properties["concept_name"],
    min_suffix_size=2,
)

# data_test = split_prefix_suffix_readable(
#     test_df,
#     case_column=properties["case_name"],
#     activity_column=properties["concept_name"],
#     min_suffix_size=2,
# )

#torch.save(data_train, '../../../perturbed_data/BPIC20/train.pkl')
torch.save(data_val, '../../../perturbed_data/BPIC20/val.pkl')
#torch.save(data_test, '../../../perturbed_data/BPIC20/test.pkl')


#display(train_df)

#extract feature info
feature_info = extract_feature_info(val_df, properties)


/home/chair/henryks_students/leon_urny/Robustness-in-suffix-prediction/robustness/perturbator/BPIC20/../event_log_loader_service/event_log_loader.py:111: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  self.df = self.df.groupby(self.case_name).apply(min_timestamp_before).reset_index(drop=True)


69873


# Create perturbed Datasets

In [8]:
# Last Event Attack
from perturbation_logic.feature_attacks import last_event_attack

# Reset data_val_copy for feature attacks
data_val_copy = data_val.copy()

# Attacks the last event of each prefix
data_pert_last = last_event_attack(
    data=data_val_copy,
    properties=properties,
    feature_info=feature_info,
    attackable_features=['org:resource','org:role'],
    num_of_features_to_attack=2,
    magnitude=0.5,
    feature_range_scope='global',
    random_seed=17
)

torch.save(data_pert_last, '../../../perturbed_data/BPIC20/last_event_attack_all.pkl')

In [11]:
# Random Event Attack
from perturbation_logic.feature_attacks import random_event_attack

# Attacks random events in each prefix with probability p
data_val_copy = data_val.copy()  
data_pert_random = random_event_attack(
    data=data_val_copy,
    properties=properties,
    feature_info=feature_info,
    attackable_features=['org:resource','org:role'],
    num_of_features_to_attack=2,
    event_attack_probability=1.0,
    magnitude=0.5,
    feature_range_scope='global',
    random_seed=17
)

torch.save(data_pert_random, '../../../perturbed_data/BPIC20/random_event_attack_all.pkl')

In [12]:
# Apply "redo last activity" augmentation to each prefix/suffix pair

data_val_copy = data_val.copy()  # Reset again
data_pert = {}
for key, (prefix_df, suffix_df) in data_val_copy.items():
    new_prefix, new_suffix = redo_last_activity_of_prefix(
        prefix_df,
        suffix_df,
        properties=properties,
    )
    data_pert[key] = (new_prefix, new_suffix)


torch.save(data_pert, '../../../perturbed_data/BPIC20/redo_pert.pkl')


In [16]:
# Apply " loop augmentation"
from perturbation_logic.structural_attacks import generate_loop_augmentation

val_loops_clean, val_loops_pert = generate_loop_augmentation(
    val_df,
    properties,
    min_suffix_size=2,
    max_matches_per_loop=3,
    save_path="../../../perturbed_data/BPIC20",
)

# Save final results
torch.save(val_loops_clean, "../../../perturbed_data/BPIC20/loop_augmentation_clean_new.pkl")
torch.save(val_loops_pert, "../../../perturbed_data/BPIC20/loop_augmentation_pert_new.pkl")
print(f"Loop augmentation: {len(val_loops_clean)} clean/pert pairs") 

Loop augmentation: 432 clean/pert pairs


# Compare changes

In [10]:
# Compare clean and perturbed datasets
from perturbation_logic.attack_impact_analyzer import highlight_feature_attack_impact

# Compare clean dataset with random event attack
highlight_feature_attack_impact(
    clean_data_path='../../../perturbed_data/BPIC20/val.pkl',
    perturbed_data_path='../../../perturbed_data/BPIC20/last_event_attack_all.pkl',
    properties=properties
)

Loading clean dataset from: ../../../perturbed_data/BPIC20/val.pkl
Loading perturbed dataset from: ../../../perturbed_data/BPIC20/last_event_attack_all.pkl

Clean dataset has 5660 cases
Perturbed dataset has 5660 cases

COMPARISON RESULTS

Case: declaration 100000, Prefix Length: 1
  [CHANGED] Found 1 event(s) with differences

  Event 0:
    org:resource:
      Clean:    STAFF MEMBER
      Perturbed: SYSTEM [CHANGED]


Case: declaration 100000, Prefix Length: 2
  ✓ No changes detected

Case: declaration 100000, Prefix Length: 3
  [CHANGED] Found 1 event(s) with differences

  Event 2:
    org:role:
      Clean:    SUPERVISOR
      Perturbed: EMPLOYEE [CHANGED]
    org:resource:
      Clean:    STAFF MEMBER
      Perturbed: SYSTEM [CHANGED]


Case: declaration 100005, Prefix Length: 1
  [CHANGED] Found 1 event(s) with differences

  Event 0:
    org:role:
      Clean:    EMPLOYEE
      Perturbed: BUDGET OWNER [CHANGED]


Case: declaration 100005, Prefix Length: 2
  [CHANGED] Found 1 ev

In [15]:
# Compare clean and perturbed datasets
from perturbation_logic.attack_impact_analyzer import highlight_structural_attack_impact

# Compare clean dataset with random event attack
highlight_structural_attack_impact(
    clean_data_path='../../../perturbed_data/BPIC20/loop_augmentation_clean_new.pkl',
    perturbed_data_path='../../../perturbed_data/BPIC20/loop_augmentation_pert_new.pkl',
    properties=properties
)

Loading clean dataset from: ../../../perturbed_data/BPIC20/loop_augmentation_clean_new.pkl
Loading perturbed dataset from: ../../../perturbed_data/BPIC20/loop_augmentation_pert_new.pkl

Clean dataset has 361 cases
Perturbed dataset has 361 cases

STRUCTURAL ATTACK COMPARISON RESULTS

Case: declaration 104453, Prefix Length: 1
Prefix
Activity_seq_clean = ['Declaration SUBMITTED by EMPLOYEE']
Activity_seq_pert = ['Declaration SUBMITTED by EMPLOYEE', 'Declaration REJECTED by ADMINISTRATION', 'Declaration REJECTED by EMPLOYEE', 'Declaration SUBMITTED by EMPLOYEE']
Case_elapsed_time_clean = [0.0]
Case_elapsed_time_pert = [0.0, 284.0, 84165.0, 84311.0]
Event_elapsed_time_clean = [nan]
Event_elapsed_time_pert = [nan, 284.0, 83881.0, 146.0]
Day_in_week_clean = [0.0]
Day_in_week_pert = [0.0, 6.0, 0.0, 0.0]
Seconds_in_day_clean = [32596.0]
Seconds_in_day_pert = [32596.0, 39050.0, 36531.0, 36677.0]

===
suffix
Activity_seq_clean = ['Declaration APPROVED by ADMINISTRATION', 'Declaration APPROVED b